# Single-LLM baseline (five SURF models)

One prompt per CSV. PANGAEA returns descriptive metadata (`name`, `description`, `keywords`) and technical metadata (`spatialCoverage`, `temporalCoverage`, `spatial`, `temporal`) in the same call. iNaturalist requests technical coverage (`spatialCoverage`, `temporalCoverage`). The user message contains the entire file as text. A dataset is sent only when that prompt plus a 4096-token completion budget fits the model's hardcoded context window. Larger files are recorded as skips and are not sent to SURF.

PANGAEA uses the baseline-specific `single_llm_pangaea_standard`. iNaturalist uses `croissant_inaturalist_standard`. The full CSV is always the proposed input: no rows or columns are removed. All 100 current iNaturalist CSVs exceed every configured context window, so this baseline records 100 context skips per model and produces no iNaturalist predictions.

Each model gets one pass before the next starts. Retryable failures remain incomplete and will be tried again on the next run.

Restart behavior:

- `{id}_single_llm.json` means the output passed Pydantic and field-completeness checks.
- `{id}_single_llm.skipped.json` means the prompt did not fit the context window, the file was unreadable, or the call hit the 10-minute timeout.
- Matching successes and skips are reused on the next run. If the CSV, prompt, or context policy changes, they are checked again. Other failures are logged and retried.

After a kernel or machine restart, use **Run All**.

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

BASE = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src/config.py').is_file()),
    None,
)
if BASE is None:
    raise FileNotFoundError('Could not locate the metadata_agent project root')
sys.path.insert(0, str(BASE))
load_dotenv(BASE / '.env')
os.environ['LLM_PROVIDER'] = 'surf'

from src.baseline.single_llm import CONTEXT_WINDOWS, DEFAULT_MODELS, PANGAEA_FULL_STANDARD_NAME, run_all_models

OUTPUTS_ROOT = BASE / 'outputs'
PROCESS_TIMEOUT_SECONDS = 600
INATURALIST_DIR_CANDIDATES = [
    BASE / 'data/experiment/inaturalist',
    BASE / 'data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs',
]


def resolve_inaturalist_dir() -> Path:
    for path in INATURALIST_DIR_CANDIDATES:
        if path.is_dir() and any(path.glob('*.csv')):
            return path
    searched = '\n  '.join(str(path) for path in INATURALIST_DIR_CANDIDATES)
    raise FileNotFoundError(f'iNaturalist data directory not found. Tried:\n  {searched}')


EXPERIMENTS = [
    {
        'dataset_label': 'PANGAEA',
        'experiment': 'pangaea',
        'standard_name': PANGAEA_FULL_STANDARD_NAME,
        'data_dir': BASE / 'data/experiment/pangaea_datasets',
        'expected': 75,
    },
    {
        'dataset_label': 'iNaturalist',
        'experiment': 'inaturalist',
        'standard_name': 'croissant_inaturalist_standard',
        'data_dir': resolve_inaturalist_dir(),
        'expected': 100,
    },
]

if not os.getenv('SURF_API_KEY'):
    raise ValueError('SURF_API_KEY is missing from metadata_agent/.env')
if not os.getenv('SURF_API_BASE'):
    raise ValueError('SURF_API_BASE is missing from metadata_agent/.env')

print(f'Project: {BASE}')
print(f'Models: {len(DEFAULT_MODELS)}')
for model in DEFAULT_MODELS:
    print(f'  {model}: context window {CONTEXT_WINDOWS[model]}')
for experiment in EXPERIMENTS:
    csv_files = sorted(experiment['data_dir'].glob('*.csv'))
    print(
        f"{experiment['dataset_label']}: {len(csv_files)} files, "
        f"standard {experiment['standard_name']}"
    )
    if len(csv_files) != experiment['expected']:
        raise ValueError(
            f"Expected {experiment['expected']} {experiment['dataset_label']} CSV files, "
            f"found {len(csv_files)}"
        )

Project: /home/com3dian/Github/metadata_agent
Models: 5
  default-text-large: context window 131072
  Qwen/Qwen3.6-35B-A3B-FP8: context window 262144
  Sehyo/Qwen3.5-122B-A10B-NVFP4: context window 262144
  mistralai/Devstral-Small-2-24B-Instruct-2512: context window 262144
  mistralai/Mistral-Small-3.2-24B-Instruct-2506: context window 131072
PANGAEA: 75 files, standard single_llm_pangaea_standard
iNaturalist: 100 files, standard croissant_inaturalist_standard


## Run or resume all five models

PANGAEA runs first, then iNaturalist. Restarting this cell skips valid successes and persistent skip records.

In [2]:
summaries = []
for experiment in EXPERIMENTS:
    print('\n' + '#' * 78)
    print(experiment['dataset_label'])
    summaries.extend(
        run_all_models(
            data_dir=experiment['data_dir'],
            outputs_root=OUTPUTS_ROOT,
            standard_name=experiment['standard_name'],
            models=DEFAULT_MODELS,
            timeout_seconds=PROCESS_TIMEOUT_SECONDS,
            experiment=experiment['experiment'],
            dataset_label=experiment['dataset_label'],
        )
    )
summaries


##############################################################################
PANGAEA
PANGAEA datasets: 75
Standard: single_llm_pangaea_standard
Models: 5

MODEL 1/5

single_llm model: default-text-large
Standard: single_llm_pangaea_standard
Output: /home/com3dian/Github/metadata_agent/outputs/single_llm_pangaea_default-text-large
Resuming with 74/75 datasets remaining; 1 already recorded.

[1/74] 899705.csv
  OK in 5.837s -> 899705_single_llm.json

[2/74] 901488.csv
  OK in 45.928s -> 901488_single_llm.json

[3/74] 901510.csv
  OK in 11.004s -> 901510_single_llm.json

[4/74] 902576.csv
  OK in 4.56s -> 902576_single_llm.json

[5/74] 906209.csv
  OK in 3.81s -> 906209_single_llm.json

[6/74] 908459.csv
  OK in 5.315s -> 908459_single_llm.json

[7/74] 913861.csv
  SKIPPED context_window (2969831 tokens, window 131072) -> 913861_single_llm.skipped.json

[8/74] 913868.csv
  SKIPPED context_window (3052235 tokens, window 131072) -> 913868_single_llm.skipped.json

[9/74] 916167.csv
  OK i

Traceback (most recent call last):
  File "/home/com3dian/Github/metadata_agent/src/baseline/single_llm/runner.py", line 598, in run_model
    else describe_csv_with_timeout(
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/com3dian/Github/metadata_agent/src/baseline/single_llm/runner.py", line 426, in describe_csv_with_timeout
    raise RuntimeError(error_queue.get())
RuntimeError: ValueError: temporalCoverage needs ordered ISO dates


  OK in 3.827s -> 937579_single_llm.json

[22/75] 937582.csv
  OK in 4.244s -> 937582_single_llm.json

[23/75] 939047.csv
  SKIPPED context_window (1330837 tokens, window 262144) -> 939047_single_llm.skipped.json

[24/75] 940249.csv
  OK in 2.536s -> 940249_single_llm.json

[25/75] 944025.csv
  OK in 3.629s -> 944025_single_llm.json

[26/75] 944039.csv
  OK in 3.59s -> 944039_single_llm.json

[27/75] 948933.csv
  OK in 2.074s -> 948933_single_llm.json

[28/75] 953747.csv
  OK in 2.126s -> 953747_single_llm.json

[29/75] 954529.csv
  OK in 1.2s -> 954529_single_llm.json

[30/75] 955790.csv
  OK in 2.919s -> 955790_single_llm.json

[31/75] 956208.csv
  OK in 3.07s -> 956208_single_llm.json

[32/75] 958774.csv
  OK in 2.778s -> 958774_single_llm.json

[33/75] 959699.csv
  OK in 2.638s -> 959699_single_llm.json

[34/75] 959883.csv
  OK in 2.558s -> 959883_single_llm.json

[35/75] 959885.csv
  OK in 2.181s -> 959885_single_llm.json

[36/75] 960522.csv
  OK in 2.346s -> 960522_single_llm.jso

Traceback (most recent call last):
  File "/home/com3dian/Github/metadata_agent/src/baseline/single_llm/runner.py", line 598, in run_model
    else describe_csv_with_timeout(
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/com3dian/Github/metadata_agent/src/baseline/single_llm/runner.py", line 426, in describe_csv_with_timeout
    raise RuntimeError(error_queue.get())
RuntimeError: ValueError: temporalCoverage needs ordered ISO dates


  OK in 3.001s -> 965137_single_llm.json

[44/75] 966669.csv
  OK in 3.117s -> 966669_single_llm.json

[45/75] 967852.csv
  OK in 2.376s -> 967852_single_llm.json

[46/75] 968881.csv
  OK in 4.867s -> 968881_single_llm.json

[47/75] 969067.csv
  OK in 17.31s -> 969067_single_llm.json

[48/75] 971414.csv
  FAILED (will retry after restart): RuntimeError: ValueError: temporalCoverage needs ordered ISO dates

[49/75] 971851.csv


...

Traceback (most recent call last):
  File "/home/com3dian/Github/metadata_agent/src/baseline/single_llm/runner.py", line 598, in run_model
    else describe_csv_with_timeout(
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/com3dian/Github/metadata_agent/src/baseline/single_llm/runner.py", line 426, in describe_csv_with_timeout
    raise RuntimeError(error_queue.get())
RuntimeError: ValueError: temporalCoverage needs ordered ISO dates


  OK in 6.817s -> 973602_single_llm.json

[56/75] 973695.csv
  OK in 14.443s -> 973695_single_llm.json

[57/75] 973907.csv
  OK in 12.935s -> 973907_single_llm.json

[58/75] 974037.csv
  OK in 7.565s -> 974037_single_llm.json

[59/75] 974295.csv
  OK in 9.157s -> 974295_single_llm.json

[60/75] 974800.csv
  OK in 6.807s -> 974800_single_llm.json

[61/75] 975181.csv
  OK in 12.604s -> 975181_single_llm.json

[62/75] 978983.csv
  OK in 8.357s -> 978983_single_llm.json

[63/75] 982395.csv
  OK in 11.474s -> 982395_single_llm.json

[64/75] 984573.csv
  OK in 13.582s -> 984573_single_llm.json

[65/75] 986963.csv
  OK in 12.196s -> 986963_single_llm.json

[66/75] 987056.csv
  OK in 9.683s -> 987056_single_llm.json

[67/75] 987068.csv
  OK in 9.919s -> 987068_single_llm.json

[68/75] 987071.csv
  OK in 9.361s -> 987071_single_llm.json

[69/75] 988976.csv
  FAILED (will retry after restart): RuntimeError: ValueError: temporalCoverage needs ordered ISO dates

[70/75] 990054.csv


Traceback (most recent call last):
  File "/home/com3dian/Github/metadata_agent/src/baseline/single_llm/runner.py", line 598, in run_model
    else describe_csv_with_timeout(
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/com3dian/Github/metadata_agent/src/baseline/single_llm/runner.py", line 426, in describe_csv_with_timeout
    raise RuntimeError(error_queue.get())
RuntimeError: ValueError: temporalCoverage needs ordered ISO dates


  OK in 14.046s -> 990054_single_llm.json

[71/75] 990157.csv
  OK in 8.642s -> 990157_single_llm.json

[72/75] 990159.csv
  OK in 8.868s -> 990159_single_llm.json

[73/75] 990164.csv
  OK in 7.946s -> 990164_single_llm.json

[74/75] 995133.csv
  OK in 6.53s -> 995133_single_llm.json

[75/75] 995327.csv
  OK in 6.692s -> 995327_single_llm.json

MODEL INCOMPLETE: 5 retryable failure(s). The notebook will continue to the next model and retry these after a restart.

##############################################################################
iNaturalist
iNaturalist datasets: 100
Standard: croissant_inaturalist_standard
Models: 5

MODEL 1/5

single_llm model: default-text-large
Standard: croissant_inaturalist_standard
Output: /home/com3dian/Github/metadata_agent/outputs/single_llm_inaturalist_default-text-large
Resuming with 100/100 datasets remaining; 0 already recorded.

[1/100] acianthus_fornicatus.csv
  SKIPPED context_window (317160 tokens, window 131072) -> acianthus_fornicatus_sin

[{'model': 'default-text-large',
  'output_dir': '/home/com3dian/Github/metadata_agent/outputs/single_llm_pangaea_default-text-large',
  'expected': 75,
  'successful': 70,
  'skipped': 5,
  'processed': 75,
  'remaining': 0},
 {'model': 'Qwen/Qwen3.6-35B-A3B-FP8',
  'output_dir': '/home/com3dian/Github/metadata_agent/outputs/single_llm_pangaea_Qwen_Qwen3.6-35B-A3B-FP8',
  'expected': 75,
  'successful': 63,
  'skipped': 4,
  'processed': 67,
  'remaining': 8},
 {'model': 'Sehyo/Qwen3.5-122B-A10B-NVFP4',
  'output_dir': '/home/com3dian/Github/metadata_agent/outputs/single_llm_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4',
  'expected': 75,
  'successful': 66,
  'skipped': 4,
  'processed': 70,
  'remaining': 5},
 {'model': 'mistralai/Devstral-Small-2-24B-Instruct-2512',
  'output_dir': '/home/com3dian/Github/metadata_agent/outputs/single_llm_pangaea_mistralai_Devstral-Small-2-24B-Instruct-2512',
  'expected': 75,
  'successful': 65,
  'skipped': 4,
  'processed': 69,
  'remaining': 6},
 {'mod